# Evaluation Demo

In this notebook we run evaluation of a trained EdgeSat-CV checkpoint on a project-owned validation dataset.

The expected evaluation data layout contains event folders with multiple Sentinel-2 observations and an accompanying change mask for the final frame.

This notebook shows how to produce qualitative and quantitative outputs from that setup.


# Get the data and prepare the environment

In [ ]:
# Remember to run this notebook with GPU
!nvidia-smi

Install these libraries:

In [ ]:
# Install libraries relevant for Colab
!pip install --quiet --upgrade gdown
!pip install --quiet hydra-core==1.1.0 kornia rasterio wandb pandas seaborn sklearn
!pip install --quiet pytorch_lightning==1.3.8
!pip install torchmetrics==0.5.1

In [ ]:
!pip install matplotlib==3.5.1 numpy==1.21.1 Pillow==8.3.1
# please restart the runtime after these!

After restarting the runtime, continue with:

In [ ]:
# Clone your project-owned EdgeSat-CV repository in a separate shell if you are running in a fresh Colab session.
# Then return here and continue with the notebook.


In [ ]:
# pretrained models
!mkdir /content/pretrained
%cd /content/pretrained

# download one model (23MB) setting with the code below (or alternatively download all the released models (138MB) from https://drive.google.com/file/d/12AwpGu7El1FWP7ErFBQ9sfuTzl2L-dT4/view?usp=sharing )
!gdown https://drive.google.com/uc?id=1LcOMmWxYSBUrH_HS747FGkhr2ZuS_y8v -O pretrained_small.zip
!unzip -q pretrained_small.zip
!rm pretrained_small.zip

%cd /content/

## Get the data

If you’re running this tutorial in **Google Colab** you need to *'add a shortcut to your Google Drive’* from our [public Google Drive folder](https://drive.google.com/drive/folders/1VEf49IDYFXGKcfvMsfh33VSiyx5MpHEn?usp=sharing) and mount that directory with the following code:

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

public_folder = '/content/drive/My Drive/EdgeSatCV_datasets'
assert os.path.exists(public_folder), "Set public_folder to a project-owned dataset directory before running this notebook."


If you're running in a **jupyter notebook** you should download the data from the [public Google Drive folder](https://drive.google.com/drive/folders/1VEf49IDYFXGKcfvMsfh33VSiyx5MpHEn?usp=sharing) manually or, alternatively you can try the `gdown` package for programatically download data from the Google Drive.

In [ ]:
# Alternatively download these with gdown:
"""
#!gdown https://drive.google.com/uc?id=1UCNnxaL9pQSkkZQx0aDEWQL0UBXPXkv0 -O fires.zip
#!gdown https://drive.google.com/uc?id=1CbNGrpK66Hos_TtOEut510k7CSHvSwkl -O landslides.zip
#!gdown https://drive.google.com/uc?id=1VP3SYgh3bj6uPa4r_bKP-5zFP3JdGin8 -O hurricanes.zip
!gdown https://drive.google.com/uc?id=1scjd4gIB_eiNS-CsOyb7Q8rYWnl9TM-L -O floods.zip
public_folder = "." # because we downloaded these locally
"""

## Unzip the data

The following two cells unzips the data and list the ground truth files with the manually identified changes.

In [ ]:
from glob import glob
files_to_extract = sorted(glob(os.path.join(public_folder, "*.zip")))
print("All:", files_to_extract)
# files_to_extract = [os.path.join(public_folder, "floods.zip")]


In [ ]:
import zipfile
from tqdm import tqdm

dataset_folder = "./datasets"
os.makedirs(dataset_folder, exist_ok=True)

for zip_files in tqdm(files_to_extract):
  with zipfile.ZipFile(zip_files, "r") as zip_ref:
    zip_ref.extractall(dataset_folder)
    zip_ref.close()

# Run the inference

We prepare the configs and run evaluation with the downloaded released model and one of the datasets (for example the *fires* event). For best results visualization we recommend logging into **Weights & Biases**, however some basic data predictions will be saved locally as well.

In [ ]:
%cd /content/EdgeSat-CV/
!ls


In [ ]:
# Edit the config programatically for Colab
p = """
---
entity: null
wandb_mode: "offline"

log_dir: "/content/outputs"
cache_dir: "/content/.cache"
"""

c = """text_file = open("config/config.yaml", "w+");text_file.write(p);text_file.close()""" 
exec(c)

In [ ]:
# Just to double-check
!cat config/config.yaml
"""
Should show:
log_dir: "/content/results"
cache_dir: "/content/cache"
"""
pass

In [ ]:
!mkdir /content/results
!mkdir /content/results/wandb/
!mkdir /content/results/cache/

In [ ]:
# ===== Parameters to adjust =====
# Note: adjust these to where you downloaded the datasets and the pretrained models

event="floods" # change this accordingly to which dataset you downloaded
checkpoint="/content/pretrained/D_train_VAE_128small/3k0vhd2o/checkpoints/epoch_00-step_29653.ckpt"
dataset_root_folder = "/content/datasets/" + event

name="VAE_128small_3k0vhd2o_epoch0___DemoInferenceOnColab"

# ===== Parameters to keep the same ======

evaluation="vae_comprehensive"
training="simple_vae"
module="deeper_vae"
plot_sequences="true"

# ========================================

!python3 -m scripts.evaluate_model \
    +dataset=floods_evaluation \
    ++dataset.root_folder=$dataset_root_folder \
    +training=$training \
    +normalisation=log_scale \
    +channels=high_res \
    +module=$module \
    +checkpoint=$checkpoint \
    +project="edgesat_demo_128small" \
    +evaluation=$evaluation \
    ++evaluation.plot_sequences=$plot_sequences \
    +name="{name}_{event}" \
    +dataset.test_overlap=[0,0] module.model_cls_args.latent_dim=128 module.model_cls_args.extra_depth_on_scale=0 module.model_cls_args.hidden_channels=[16,32,64] \
    ++evaluation.save_plots_locally=true \
    training.num_workers=2 training.batch_size_train=64 training.batch_size_valid=64 training.batch_size_test=64

    # the last few args set the same architecture as the one we used when training
    # and some hw specs settings:
    # training.num_workers=2 training.batch_size_train=64 training.batch_size_valid=64 training.batch_size_test=64

In [ ]:
# Time estimation: for the "floods" dataset (~2.45GB), the validation takes approximately
# 40 mins because we are running and comparing many methods. You can also access the intermediate
# results in the outputs/ folder or on wandb directly...


# 3 Results

Results are best to be observed on the generated Wandb page, however there are also local files saved which we can observe.

In [ ]:
!ls outputs/*.csv

Note that we have results corresponding to all of the above csv files. Just select one by setting ``` index = 0 ``` (1,2, etc...).



In [ ]:
import matplotlib.pyplot as plt

# Select the index of the desired event
index = 1
#index = 0

# Event visualization
plt.figure(figsize=(8, 6), dpi=150)

plt.subplot(1, 2, 1)
img = plt.imread("./outputs/"+str(index).zfill(3)+"_1_before.png")
plt.imshow(img)
plt.gca().set_title('Before')

plt.subplot(1, 2, 2)
img = plt.imread("./outputs/"+str(index).zfill(3)+"_2_after.png")
plt.imshow(img)
plt.gca().set_title('After')

plt.show()

In [ ]:
# Event visualization
from IPython.display import Image
print('Ground truth')
Image("./outputs/"+str(index).zfill(3)+"_3_change mask.png",
      height=400)

In [ ]:
# Event visualization
from IPython.display import Image
print('Baseline (pixel values in cosine distance, with memory 3)')
Image("./outputs/"+str(index).zfill(3)+"_10_cos_pixel | memory 3 | 32x32 - mean.png",
      height=400)

In [ ]:
print('EdgeSat-CV (cosine distance of learned embedded tiles, with memory 3)')
Image("./outputs/"+str(index).zfill(3)+"_11_cos_emb | memory 3 | 32x32 - mean.png",
      height=400)

Quantitative results (for the selected single index) are available in the csv files. To access the overall results for this subset see the *'Detection technique summary statistics'* table on wandb.

In [ ]:
import pandas as pd 
df=pd.read_csv("./outputs/"+str(index).zfill(3)+"_stats.csv")
df

Optionally download the results:

In [ ]:
!zip -r edgesat_predictions.zip outputs/

In [ ]:
!ls *.zip -luah

In [ ]:
from google.colab import files
files.download('edgesat_predictions.zip') 